# GraphRAG 与混合检索 — 第三节
## GraphRAG & Hybrid Retrieval — Part 3

**核心主题 / Core Topic：**

> 向量检索 + BM25 关键词检索 + 图谱检索，通过 **RRF（倒数排名融合）** 合并结果  
> Combining vector search, BM25 keyword search, and graph-based lookup using **Reciprocal Rank Fusion (RRF)**

---

### 三路检索融合架构

```
                   ┌─────────────┐
                   │    Query    │
                   └──────┬──────┘
          ┌───────────────┼───────────────┐
          ▼               ▼               ▼
   向量检索(Vector)   BM25 检索      图谱检索(Graph)
   语义相似度          精确词元匹配    多跳关系遍历
          └───────────────┼───────────────┘
                          ▼
                   RRF 融合排序
                   score = Σ 1/(k + rank)
                          ▼
                   最终排序结果
```

### 三类查询各自的最优检索方式

| 查询类型 | 示例 | 最优方式 |
|----------|------|----------|
| 语义查询 | iPhone 15 Pro Max 用什么芯片？ | ✅ 向量检索 |
| 多跳推理 | Apple CEO 的大学在哪个城市？ | ✅ 图谱检索 |
| 精确关键词 | September 4 1998 | ✅ BM25 |

---

**安装依赖 / Install:**
```bash
pip install chromadb sentence-transformers rank-bm25
```

## Step 1 — 导入依赖 / Import Dependencies

In [1]:
from __future__ import annotations

import chromadb
from chromadb.utils import embedding_functions
from rank_bm25 import BM25Okapi

print("✅ 依赖导入成功")

✅ 依赖导入成功


## Step 2 — 文档语料库 / Document Corpus

相比第一节，语料库更丰富：每个实体的信息拆分为**独立文档**，模拟真实 RAG 知识库。

例如 Tim Cook 被拆为：
- `doc_1`：Tim Cook 是 Apple CEO
- `doc_2`：Tim Cook 在 Auburn University 读书
- `doc_3`：Auburn University 在 Alabama 的 Auburn 市

In [2]:
DOCUMENTS: list[str] = [
    "Apple Inc. was founded on April 1, 1976 by Steve Jobs, Steve Wozniak, and Ronald Wayne.",   # doc_0
    "Tim Cook became the CEO of Apple in August 2011, succeeding Steve Jobs.",                   # doc_1
    "Tim Cook studied Industrial Engineering at Auburn University in Auburn, Alabama.",           # doc_2
    "Auburn University is a public research university located in Auburn, Alabama.",              # doc_3
    "Apple's headquarters is located in Cupertino, California at Apple Park.",                   # doc_4
    "The iPhone 15 Pro Max features a titanium frame, USB-C port, and an A17 Pro chip.",         # doc_5
    "Google was incorporated on September 4, 1998 by Larry Page and Sergey Brin.",               # doc_6
    "Sundar Pichai is the CEO of Google and its parent company Alphabet Inc.",                   # doc_7
    "Sundar Pichai attended Stanford University for his MS in Materials Science.",               # doc_8
    "Stanford University is situated in Stanford, California near Palo Alto.",                   # doc_9
    "Microsoft was founded on April 4, 1975 by Bill Gates and Paul Allen in Redmond, Washington.", # doc_10
    "Satya Nadella has served as Microsoft's CEO since February 2014.",                          # doc_11
    "Azure Cloud is Microsoft's flagship cloud computing platform launched in 2010.",             # doc_12
    "The MacBook Pro M3 was released in October 2023 with up to 22-hour battery life.",          # doc_13
    "Google Search was launched in 1998 and processes over 8.5 billion queries per day.",        # doc_14
]

DOC_IDS: list[str] = [f"doc_{i}" for i in range(len(DOCUMENTS))]

print(f"语料库共 {len(DOCUMENTS)} 条文档")
for i, doc in enumerate(DOCUMENTS):
    print(f"  [doc_{i:02d}] {doc[:80]}{'…' if len(doc) > 80 else ''}")

语料库共 15 条文档
  [doc_00] Apple Inc. was founded on April 1, 1976 by Steve Jobs, Steve Wozniak, and Ronald…
  [doc_01] Tim Cook became the CEO of Apple in August 2011, succeeding Steve Jobs.
  [doc_02] Tim Cook studied Industrial Engineering at Auburn University in Auburn, Alabama.
  [doc_03] Auburn University is a public research university located in Auburn, Alabama.
  [doc_04] Apple's headquarters is located in Cupertino, California at Apple Park.
  [doc_05] The iPhone 15 Pro Max features a titanium frame, USB-C port, and an A17 Pro chip…
  [doc_06] Google was incorporated on September 4, 1998 by Larry Page and Sergey Brin.
  [doc_07] Sundar Pichai is the CEO of Google and its parent company Alphabet Inc.
  [doc_08] Sundar Pichai attended Stanford University for his MS in Materials Science.
  [doc_09] Stanford University is situated in Stanford, California near Palo Alto.
  [doc_10] Microsoft was founded on April 4, 1975 by Bill Gates and Paul Allen in Redmond, …
  [doc_11] Satya Nadel

## Step 3 — 模拟图谱索引 / Simulated Graph Index

**图谱索引**：将实体/概念映射到可通过图遍历找到的文档列表。

生产环境中，这里会替换为真实的 **Cypher 多跳查询**（对接 Neo4j）。

例如查询 `"apple ceo university city"` → 触发关系链：
```
Apple → CEO(Tim Cook) [doc_1] → Auburn University [doc_2] → Auburn, AL [doc_3]
```

In [3]:
# Maps an entity/concept to the documents reachable via graph traversal.
# In production: replace with Cypher multi-hop queries on Neo4j.
GRAPH_INDEX: dict[str, list[int]] = {
    "apple ceo university city":      [1, 2, 3],   # Tim Cook → Auburn Univ → Auburn, AL
    "google ceo university location": [7, 8, 9],   # Sundar → Stanford → Stanford, CA
    "microsoft ceo":                  [10, 11],
    "apple products":                 [5, 13],
    "google products":                [6, 14],
    "microsoft products":             [12],
}

print("图谱索引（关键词 → 可触达文档）:")
for key, indices in GRAPH_INDEX.items():
    docs_preview = [f"doc_{i}" for i in indices]
    print(f"  '{key}' → {docs_preview}")

图谱索引（关键词 → 可触达文档）:
  'apple ceo university city' → ['doc_1', 'doc_2', 'doc_3']
  'google ceo university location' → ['doc_7', 'doc_8', 'doc_9']
  'microsoft ceo' → ['doc_10', 'doc_11']
  'apple products' → ['doc_5', 'doc_13']
  'google products' → ['doc_6', 'doc_14']
  'microsoft products' → ['doc_12']


## Step 4 — 构建向量索引 & BM25 索引 / Build Indexes

In [4]:
def build_vector_store() -> chromadb.Collection:
    """Build an in-memory ChromaDB collection."""
    client = chromadb.Client()
    ef = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )
    col = client.create_collection(name="hybrid_demo", embedding_function=ef)
    col.add(documents=DOCUMENTS, ids=DOC_IDS)
    return col


def build_bm25_index() -> BM25Okapi:
    """Build a BM25 index over the corpus."""
    tokenised = [doc.lower().split() for doc in DOCUMENTS]
    return BM25Okapi(tokenised)


print("正在构建索引（首次运行会下载 MiniLM 模型）…")
collection = build_vector_store()
bm25 = build_bm25_index()
print(f"✅ 向量索引: {collection.count()} 条文档")
print(f"✅ BM25 索引: {len(DOCUMENTS)} 条文档")

正在构建索引（首次运行会下载 MiniLM 模型）…
✅ 向量索引: 15 条文档
✅ BM25 索引: 15 条文档


## Step 5 — 三路检索函数 / Three Search Functions

### `graph_search` 算法图解 / Graph RAG Algorithm Walkthrough

`graph_search` **模拟** GraphRAG 的核心能力：用**知识图谱的多跳关系**，把分散在多篇文档里的信息串起来召回。  
生产环境会把 `GRAPH_INDEX` 替换成 Neo4j 上的 **Cypher 多跳查询**。

---

#### 1. 整体思路：从「图谱」到「文档」

```mermaid
flowchart TB
    subgraph Build["离线建图（预处理）"]
        D1["doc_1: Tim Cook 是 Apple CEO"]
        D2["doc_2: Tim Cook 在 Auburn University 读书"]
        D3["doc_3: Auburn University 在 Auburn, AL"]
    end

    subgraph Graph["知识图谱（实体 + 关系）"]
        A((Apple))
        T((Tim Cook))
        U((Auburn University))
        C((Auburn, AL))
        A -->|"CEO"| T
        T -->|"studied at"| U
        U -->|"located in"| C
    end

    subgraph Index["GRAPH_INDEX（预计算路径 → 文档）"]
        K["'apple ceo university city'"]
        V["[1, 2, 3]"]
        K --> V
    end

    D1 -.-> T
    D2 -.-> U
    D3 -.-> C
    Graph --> Index
```

**本 notebook 的简化**：不实时跑图遍历，而是把常见多跳路径**预存**成 `GRAPH_INDEX`：

| 图谱 Key | 含义（多跳链） | 触达文档 |
|----------|----------------|----------|
| `apple ceo university city` | Apple → CEO → 大学 → 城市 | doc_1, doc_2, doc_3 |
| `google ceo university location` | Google → CEO → 大学 → 位置 | doc_7, doc_8, doc_9 |

---

#### 2. `graph_search` 算法流程

```mermaid
flowchart TD
    Q["输入 Query"]
    L["query_lower = query.lower()"]
    Loop["遍历 GRAPH_INDEX 每个 (key, indices)"]
    Overlap["计算 token 重叠度<br/>overlap = Σ(key 中 token 是否出现在 query)"]
    Check{"overlap ≥ 2 ?"}
    Match["matched.extend(indices)"]
    Skip["跳过此 key"]
    Dedup["去重保序"]
    TopN["取前 n 条，返回 (doc_idx, doc_text)"]

    Q --> L --> Loop --> Overlap --> Check
    Check -->|是| Match --> Loop
    Check -->|否| Skip --> Loop
    Loop --> Dedup --> TopN
```

核心逻辑：

```python
for key, indices in GRAPH_INDEX.items():
    overlap = sum(1 for token in key.split() if token in query_lower)
    if overlap >= 2:          # 阈值：至少 2 个 token 命中才触发
        matched.extend(indices)
# 去重保序 → 截断 top-n
```

---

#### 3. 示例：Query B 多跳推理

**Query**: `"Which city is Apple CEO's university located in?"`

**Step 1 — Token 匹配打分**

| GRAPH_INDEX Key | 命中 token | overlap | 是否触发 |
|-----------------|------------|---------|----------|
| `apple ceo university city` | apple, ceo, university, city | **4** | ✅ |
| `google ceo university location` | ceo, university | 2 | ✅ |
| `microsoft ceo` | ceo | 1 | ❌ |
| `apple products` | apple | 1 | ❌ |

阈值 `≥ 2` 的作用：**避免单个词（如 "apple"）误触发整条链**。

**Step 2 — 图谱多跳链**

```
Query: "Apple CEO's university ... city"
              │
              ▼  命中 key: "apple ceo university city"
┌─────────────────────────────────────────────────────────┐
│   [Apple] ──CEO──▶ [Tim Cook] ──studied──▶ [Auburn Univ] ──in──▶ [Auburn, AL]
│      │                  │                        │                    │
│   (doc_0)            doc_1                   doc_2                 doc_3
│   公司信息         CEO 身份              在哪读大学              大学所在城市
└─────────────────────────────────────────────────────────┘
```

**Step 3 — 返回结果**

Graph Search 一次性召回整条推理链：`doc_1 → doc_2 → doc_3`  
向量/BM25 单次检索很难同时把这三篇文档排齐。

---

#### 4. 三路检索对比 & 在 RRF 中的位置

| 能力 | 向量 | BM25 | Graph (`graph_search`) |
|------|------|------|------------------------|
| 语义理解 | ✅ 强 | ❌ 弱 | ❌ 弱（靠 key 匹配） |
| 精确关键词 | ❌ 弱 | ✅ 强 | ❌ 弱 |
| **多跳关系** | ❌ 弱 | ❌ 弱 | ✅ **强** |

```mermaid
flowchart TB
    Q["Query"]
    Q --> V["vector_search"]
    Q --> B["bm25_search"]
    Q --> G["graph_search"]

    V --> RRF["RRF 融合 score = Σ 1/(60 + rank)"]
    B --> RRF
    G --> RRF
    RRF --> F["最终排序结果"]
```

---

#### 5. 模拟 vs 生产

| 环节 | Notebook 模拟 | 生产 GraphRAG |
|------|---------------|---------------|
| 建图 | 手写 `GRAPH_INDEX` | LLM 抽取实体关系 → Neo4j |
| 查询 | token 重叠 ≥ 2 | Cypher 多跳遍历 |
| 召回 | 固定 doc 索引列表 | 沿边遍历，返回关联 chunk |

> **一句话**：`graph_search` = 用「预存的多跳关系路径」做查询路由；当 query 与某条路径的关键词重叠足够，就召回该路径上所有文档，解决跨文档推理问题。


In [5]:
def vector_search(collection: chromadb.Collection, query: str, n: int = 5) -> list[tuple[int, str]]:
    """Return (doc_index, doc_text) pairs ranked by vector similarity."""
    results = collection.query(query_texts=[query], n_results=n)
    ids = results["ids"][0]
    docs = results["documents"][0]
    return [(int(doc_id.split("_")[1]), doc) for doc_id, doc in zip(ids, docs)]


def bm25_search(index: BM25Okapi, query: str, n: int = 5) -> list[tuple[int, str]]:
    """Return (doc_index, doc_text) pairs ranked by BM25 score."""
    tokens = query.lower().split()
    scores = index.get_scores(tokens)
    ranked_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return [(i, DOCUMENTS[i]) for i in ranked_indices[:n]]


def graph_search(query: str, n: int = 5) -> list[tuple[int, str]]:
    """
    Simulate a graph traversal lookup.
    Production: run Cypher multi-hop query and return matching chunks.
    """
    query_lower = query.lower()
    matched: list[int] = []
    for key, indices in GRAPH_INDEX.items():
        # 按关键词 token 重叠度打分，≥2 则触发
        overlap = sum(1 for token in key.split() if token in query_lower)
        if overlap >= 2:
            matched.extend(indices)

    # 去重保序
    seen: set[int] = set()
    unique: list[int] = []
    for idx in matched:
        if idx not in seen:
            seen.add(idx)
            unique.append(idx)

    return [(i, DOCUMENTS[i]) for i in unique[:n]]


print("✅ 三路检索函数定义完成")

✅ 三路检索函数定义完成


## Step 6 — RRF 融合算法 / Reciprocal Rank Fusion

**RRF 公式**（来自 Cormack et al., 2009）：

$$
\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}
$$

- $R$：所有检索器的结果列表
- $k$：平滑常数，默认 60，防止头部排名过度主导
- $\text{rank}_r(d)$：文档 $d$ 在检索器 $r$ 中的排名（从 1 开始）

**核心优势**：
- 不需要对不同检索器的分数做归一化
- 只关心**排名**而非原始分数
- 在多路融合中被多次检索的文档会获得更高的综合分

### `reciprocal_rank_fusion` 算法图解 / RRF Algorithm Walkthrough

RRF 把**多路检索器的排名列表**融合成一份最终排序，**只用排名、不用原始分数**，因此向量/BM25/Graph 三路分数尺度不同也能直接合并。

---

#### 1. 整体流程

```mermaid
flowchart TB
    subgraph Input["输入：3 路排名列表"]
        V["Vector  ranked list"]
        B["BM25    ranked list"]
        G["Graph   ranked list"]
    end

    subgraph Score["逐路累加 RRF 分数"]
        F["对每个 (doc_idx, rank)<br/>score += 1 / (k + rank)"]
    end

    subgraph Output["输出"]
        S["按 RRF 总分降序排列"]
        R["(rrf_score, doc_idx, doc_text)"]
    end

    V --> F
    B --> F
    G --> F
    F --> S --> R
```

对应代码三步：

```python
# ① 遍历每路排名，累加 1/(k+rank)
for ranked in results_list:
    for rank, (doc_idx, doc_text) in enumerate(ranked, start=1):
        scores[doc_idx] += 1.0 / (k + rank)

# ② 组装 (score, idx, text)  ③ 按 score 降序
fused.sort(key=lambda x: x[0], reverse=True)
```

---

#### 2. 核心公式与 k 的作用

$$
\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}
$$

| 符号 | 含义 |
|------|------|
| $R$ | 所有检索器（Vector、BM25、Graph） |
| $\text{rank}_r(d)$ | 文档 $d$ 在检索器 $r$ 中的排名，**从 1 开始** |
| $k$ | 平滑常数，默认 **60**（Cormack et al., 2009） |

**k 为何取 60？** 拉大排名差距，避免第 1 名过度垄断：

```
rank=1  →  1/(60+1)  = 0.0164
rank=2  →  1/(60+2)  = 0.0161   ← 与第 1 名差距很小
rank=10 →  1/(60+10) = 0.0143
```

若 $k=0$，第 1 名得分 1.0、第 2 名 0.5，头部优势过大；$k=60$ 使各路排名更「民主」地贡献分数。

---

#### 3. 单文档打分示意

以 **doc_1**（Tim Cook CEO）在 Query B 中为例：

```
                rank   贡献分 1/(60+rank)
Vector  列表:   3  →  1/63 = 0.0159
BM25    列表:  (未出现) →  0
Graph   列表:   1  →  1/61 = 0.0164
                              ─────────
                    RRF(doc_1) = 0.0323  ← 两路命中，分数叠加
```

**关键规则**：文档**只在出现的路**上得分；被多路同时召回的文档获得**加成**。

---

#### 4. Query B 完整融合示例

**Query**: `"Which city is Apple CEO's university located in?"`

**三路排名（top 3）**

| rank | Vector | BM25 | Graph |
|------|--------|------|-------|
| 1 | doc_4 | doc_4 | **doc_1** |
| 2 | doc_0 | doc_3 | **doc_2** |
| 3 | **doc_1** | doc_9 | **doc_3** |

**RRF 累加（k=60，节选）**

| doc | Vector 贡献 | BM25 贡献 | Graph 贡献 | **RRF 总分** |
|-----|------------|-----------|------------|-------------|
| doc_1 | 1/63=0.0159 | — | 1/61=0.0164 | **0.0323** → 融合第 1 |
| doc_4 | 1/61=0.0164 | 1/61=0.0164 | — | **0.0328** → 融合第 2 |
| doc_3 | — | 1/62=0.0161 | 1/63=0.0159 | **0.0320** → 融合第 3 |
| doc_2 | — | — | 1/62=0.0161 | **0.0161** → Graph 独命中 |

```
                    Vector          BM25           Graph
                      │              │               │
doc_4 (Cupertino)  rank=1 ────── rank=1 ────────── (无)
doc_1 (Tim CEO)    rank=3 ────── (无) ────────── rank=1  ← 两路命中，RRF 最高
doc_3 (Auburn Uni)  (无) ────── rank=2 ────────── rank=3  ← 两路命中
doc_2 (Tim 读书)    (无) ────── (无) ────────── rank=2  ← Graph 独贡献
                      │              │               │
                      └──────────────┴───────────────┘
                                     ▼
                            RRF 分数累加 & 重排
                                     ▼
                    doc_1 > doc_4 > doc_3 > doc_0 > doc_2
```

Graph 把 doc_1/doc_2/doc_3 推入候选；doc_1 同时在 Vector 第 3，**双路加成**排到融合榜首。

---

#### 5. 为何不用原始分数加权？

```mermaid
flowchart LR
    subgraph Problem["直接加权的问题"]
        VS["Vector score: 0.82"]
        BS["BM25 score: 12.5"]
        GS["Graph: 无分数"]
        P["尺度不同，无法直接相加"]
    end

    subgraph RRF["RRF 的思路"]
        R1["只看排名: 第1、第2、第3…"]
        R2["1/(k+rank) 天然归一化"]
        R3["任意数量检索器可扩展"]
    end

    Problem --> RRF
```

| 对比 | 分数加权融合 | RRF |
|------|-------------|-----|
| 需要归一化 | ✅ 必须 | ❌ 不需要 |
| 依赖原始分数 | ✅ | ❌ 只用 rank |
| 某路无分数（如 Graph） | 难处理 | ✅ 自然兼容 |
| 多路同时命中 | 需调权重 | ✅ 自动加成 |

---

#### 6. 算法复杂度

- 时间：$O(\sum |L_i|)$，$L_i$ 为第 $i$ 路结果长度；本 demo 三路各 top-5 → 最多 15 次累加
- 空间：$O(D)$，$D$ 为去重后的文档数

> **一句话**：RRF = 把每路排名的倒数「投票」累加；排得越靠前、被越多路召回，总分越高。


In [6]:
def reciprocal_rank_fusion(
    results_list: list[list[tuple[int, str]]],
    k: int = 60,
) -> list[tuple[float, int, str]]:
    """
    Reciprocal Rank Fusion: fuse multiple ranked result lists.

    Args:
        results_list: Each inner list is a ranked sequence of (doc_index, doc_text).
        k: Smoothing constant (default 60, from the original RRF paper).

    Returns:
        Sorted list of (rrf_score, doc_index, doc_text), highest score first.
    """
    scores: dict[int, float] = {}
    texts: dict[int, str] = {}

    for ranked in results_list:
        for rank, (doc_idx, doc_text) in enumerate(ranked, start=1):
            scores[doc_idx] = scores.get(doc_idx, 0.0) + 1.0 / (k + rank)
            texts[doc_idx] = doc_text

    fused = [(score, idx, texts[idx]) for idx, score in scores.items()]
    fused.sort(key=lambda x: x[0], reverse=True)
    return fused


print("✅ RRF 融合函数定义完成")

# 展示 k=60 时不同排名对应的 RRF 分数
print("\nRRF 分数示例 (k=60):")
for rank in [1, 2, 3, 5, 10, 20]:
    print(f"  rank={rank:2d} → score={1/(60+rank):.6f}")

✅ RRF 融合函数定义完成

RRF 分数示例 (k=60):
  rank= 1 → score=0.016393
  rank= 2 → score=0.016129
  rank= 3 → score=0.015873
  rank= 5 → score=0.015385
  rank=10 → score=0.014286
  rank=20 → score=0.012500


## Step 7 — 展示辅助函数 / Display Helpers

In [7]:
def show_results(label: str, results: list[tuple[int, str]], top: int = 3) -> None:
    print(f"  [{label}]")
    for rank, (idx, doc) in enumerate(results[:top], 1):
        print(f"    {rank}. [doc_{idx:02d}] {doc[:88]}{'…' if len(doc) > 88 else ''}")


def show_fused(fused: list[tuple[float, int, str]], top: int = 5) -> None:
    print("  [RRF Fused Result]")
    for rank, (score, idx, doc) in enumerate(fused[:top], 1):
        print(f"    {rank}. score={score:.4f} [doc_{idx:02d}] {doc[:83]}{'…' if len(doc) > 83 else ''}")


def run_query(label: str, query: str) -> None:
    print(f"\nQuery: \"{query}\"")
    vec_results   = vector_search(collection, query)
    bm25_results  = bm25_search(bm25, query)
    graph_results = graph_search(query)

    show_results("Vector Search", vec_results)
    show_results("BM25 Search  ", bm25_results)
    show_results("Graph Search ", graph_results)
    print()
    fused = reciprocal_rank_fusion([vec_results, bm25_results, graph_results])
    show_fused(fused)


print("✅ 展示函数定义完成")

✅ 展示函数定义完成


---
## 🔍 查询 A：语义查询 / Query A — Semantic (Vector-Friendly)

**问题**：iPhone 15 Pro Max 用什么芯片？

**分析**：
- 向量检索：能理解「chip」「芯片」等语义，召回 `doc_5`
- BM25：依赖精确词元，`iPhone 15 Pro Max` 在语料中存在，也可命中
- Graph：查询词与图谱 key 重叠不足，基本无结果

**预期**：向量检索主导，RRF 综合排名 `doc_5` 靠前

In [8]:
print("=" * 65)
print("QUERY A — Single-Hop (semantic, vector-friendly)")
print("=" * 65)
run_query(
    label="single-hop",
    query="What chip does the iPhone 15 Pro Max use?",
)

QUERY A — Single-Hop (semantic, vector-friendly)

Query: "What chip does the iPhone 15 Pro Max use?"
  [Vector Search]
    1. [doc_05] The iPhone 15 Pro Max features a titanium frame, USB-C port, and an A17 Pro chip.
    2. [doc_01] Tim Cook became the CEO of Apple in August 2011, succeeding Steve Jobs.
    3. [doc_04] Apple's headquarters is located in Cupertino, California at Apple Park.
  [BM25 Search  ]
    1. [doc_05] The iPhone 15 Pro Max features a titanium frame, USB-C port, and an A17 Pro chip.
    2. [doc_13] The MacBook Pro M3 was released in October 2023 with up to 22-hour battery life.
    3. [doc_01] Tim Cook became the CEO of Apple in August 2011, succeeding Steve Jobs.
  [Graph Search ]

  [RRF Fused Result]
    1. score=0.0328 [doc_05] The iPhone 15 Pro Max features a titanium frame, USB-C port, and an A17 Pro chip.
    2. score=0.0320 [doc_01] Tim Cook became the CEO of Apple in August 2011, succeeding Steve Jobs.
    3. score=0.0318 [doc_13] The MacBook Pro M3 was re

---
## 🔗 查询 B：多跳推理 / Query B — Multi-Hop (Graph-Friendly)

**问题**：Apple CEO 的大学在哪个城市？

**分析**：
- 向量检索：单次查询语义最近的文档，难以同时关联三个跳
- BM25：词元 `apple`、`ceo`、`university`、`city` 分散在多个文档
- Graph：命中 `"apple ceo university city"` 关键词组合，直接返回 `[doc_1, doc_2, doc_3]`

**预期**：图谱检索发挥关键作用，RRF 融合后 `doc_2`（Auburn University）排名显著提升

In [9]:
print("=" * 65)
print("QUERY B — Multi-Hop (requires graph traversal)")
print("=" * 65)
run_query(
    label="multi-hop",
    query="Which city is Apple CEO's university located in?",
)

QUERY B — Multi-Hop (requires graph traversal)

Query: "Which city is Apple CEO's university located in?"
  [Vector Search]
    1. [doc_04] Apple's headquarters is located in Cupertino, California at Apple Park.
    2. [doc_00] Apple Inc. was founded on April 1, 1976 by Steve Jobs, Steve Wozniak, and Ronald Wayne.
    3. [doc_01] Tim Cook became the CEO of Apple in August 2011, succeeding Steve Jobs.
  [BM25 Search  ]
    1. [doc_04] Apple's headquarters is located in Cupertino, California at Apple Park.
    2. [doc_03] Auburn University is a public research university located in Auburn, Alabama.
    3. [doc_09] Stanford University is situated in Stanford, California near Palo Alto.
  [Graph Search ]
    1. [doc_01] Tim Cook became the CEO of Apple in August 2011, succeeding Steve Jobs.
    2. [doc_02] Tim Cook studied Industrial Engineering at Auburn University in Auburn, Alabama.
    3. [doc_03] Auburn University is a public research university located in Auburn, Alabama.

  [RRF Fus

---
## 📅 查询 C：精确关键词 / Query C — Exact Keyword (BM25-Friendly)

**问题**：`September 4 1998`

**分析**：
- 向量检索：日期嵌入语义相近但不精确，可能混淆「1976」「1975」
- BM25：精确匹配 `September`、`4`、`1998`，直接命中 `doc_6`
- Graph：无重叠触发

**预期**：BM25 主导，RRF 融合后 `doc_6`（Google 成立日期）排名第一

In [10]:
print("=" * 65)
print("QUERY C — Exact Keyword (BM25-friendly)")
print("=" * 65)
run_query(
    label="exact-keyword",
    query="September 4 1998",
)

QUERY C — Exact Keyword (BM25-friendly)

Query: "September 4 1998"
  [Vector Search]
    1. [doc_06] Google was incorporated on September 4, 1998 by Larry Page and Sergey Brin.
    2. [doc_14] Google Search was launched in 1998 and processes over 8.5 billion queries per day.
    3. [doc_10] Microsoft was founded on April 4, 1975 by Bill Gates and Paul Allen in Redmond, Washingt…
  [BM25 Search  ]
    1. [doc_06] Google was incorporated on September 4, 1998 by Larry Page and Sergey Brin.
    2. [doc_14] Google Search was launched in 1998 and processes over 8.5 billion queries per day.
    3. [doc_00] Apple Inc. was founded on April 1, 1976 by Steve Jobs, Steve Wozniak, and Ronald Wayne.
  [Graph Search ]

  [RRF Fused Result]
    1. score=0.0328 [doc_06] Google was incorporated on September 4, 1998 by Larry Page and Sergey Brin.
    2. score=0.0323 [doc_14] Google Search was launched in 1998 and processes over 8.5 billion queries per day.
    3. score=0.0313 [doc_00] Apple Inc. was foun

---
## 📊 总结 / Takeaway

| 检索方式 | 最优场景 | 弱项 |
|----------|----------|------|
| 向量检索 (Vector) | 语义相似，同义词/改写 | 精确词元、多跳推理 |
| BM25 | 精确词元、日期/代码 | 语义理解 |
| 图谱检索 (Graph) | 多跳关系推理 | 模糊语义查询 |
| **RRF 融合** | **覆盖所有场景** | 依赖各路质量 |

### RRF 核心价值

```
• 不需要归一化各路分数 → 简单且鲁棒
• 被多路检索器命中的文档 → 获得加成
• k=60 平滑 → 避免头部排名垄断
• 适应任意数量的检索器
```

**下一节将实现完整的端到端 GraphRAG Pipeline，引入 LLM 生成最终答案。**